# 01 - Fetch Papers

In this notebook, I build the paper level dataset for the 2017-2025
analysis window.

PACMPL papers come from OpenAlex using the PACMPL ISSN. PLDI is handled in
two parts: PLDI 2017-2022 comes from DBLP DOI lists followed by OpenAlex
lookups, and PLDI 2023-2025 comes from the PACMPL issue `PLDI`.

I keep `OOPSLA1` and `OOPSLA2` separate. This matters because OOPSLA changed
to a two round structure from 2022 onward.


## 1 - Setup

In [ ]:
import json
import os
import re
import socket
import time

import pyalex
import pandas as pd
import requests

from pathlib import Path
from pyalex import Works


In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_2_data_dir = project_folder / "step_2_data"
step_2_artifacts_dir = project_folder / "step_2_artifacts"
raw_pacmpl_dir = step_2_data_dir / "raw" / "pacmpl_papers"
raw_pldi_dir = step_2_data_dir / "raw" / "pldi_papers"
prepared_pacmpl_dir = step_2_data_dir / "prepared" / "pacmpl_papers"
prepared_pldi_dir = step_2_data_dir / "prepared" / "pldi_papers"
prepared_all_dir = step_2_data_dir / "prepared" / "all_papers"
summary_tables_dir = step_2_artifacts_dir / "summary_tables"
dependency_tables_dir = step_2_artifacts_dir / "dependency_tables"
check_tables_dir = step_2_artifacts_dir / "check_tables"

raw_pacmpl_dir.mkdir(parents=True, exist_ok=True)
raw_pldi_dir.mkdir(parents=True, exist_ok=True)
prepared_pacmpl_dir.mkdir(parents=True, exist_ok=True)
prepared_pldi_dir.mkdir(parents=True, exist_ok=True)
prepared_all_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)


def load_env_file(path):
    if not path.exists():
        return False
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
    return True


loaded_env_files = [
    str(path.relative_to(project_folder))
    for path in [project_folder / ".env", project_folder / "key.env"]
    if load_env_file(path)
]

REFRESH_DATA = overwrite_data
FETCH_SLEEP = 0.10
DBLP_SLEEP = 5.0
DBLP_MAX_RETRIES = 5
DBLP_RETRY_WAIT = 45
ANALYSIS_START = 2017
ANALYSIS_END = 2025

socket.setdefaulttimeout(30)

if os.environ.get("OPENALEX_API_KEY"):
    pyalex.config.api_key = os.environ["OPENALEX_API_KEY"]

pyalex.config.max_retries = 2
pyalex.config.retry_backoff_factor = 0.5
pyalex.config.retry_http_codes = [429, 500, 502, 503, 504]

print(project_folder)
print(f"Run mode: {run_mode}")
print(f"Loaded local env files: {loaded_env_files}")
print(f"OpenAlex API key configured: {bool(os.environ.get('OPENALEX_API_KEY'))}")


## 2 - Helper Functions

In [ ]:
def short_openalex_id(value):
    if not value:
        return None
    return str(value).rstrip("/").rsplit("/", 1)[-1]


def normalize_doi(doi):
    if not isinstance(doi, str) or not doi.strip():
        return None

    doi = doi.strip()
    doi = doi.replace("http://doi.org/", "").replace("https://doi.org/", "")
    return "https://doi.org/" + doi.lower()


def doi_cache_key(doi):
    doi = normalize_doi(doi)
    if doi is None:
        return None
    doi = doi.replace("https://doi.org/", "")
    return doi.replace("/", "_").replace(":", "_")


def short_orcid(orcid):
    if not isinstance(orcid, str) or not orcid.strip():
        return None
    return orcid.rstrip("/").rsplit("/", 1)[-1]


def flatten_authorships(work):
    authorships = []

    for authorship in work.get("authorships") or []:
        author = authorship.get("author") or {}

        authorships.append({
            "author_id": short_openalex_id(author.get("id")),
            "author_name": author.get("display_name"),
            "orcid": short_orcid(author.get("orcid") or authorship.get("raw_orcid")),
            "position": authorship.get("author_position") or authorship.get("position"),
            "is_corresponding": authorship.get("is_corresponding"),
        })

    return authorships


def volume_to_conference_year(volume):
    try:
        return 2016 + int(volume)
    except (TypeError, ValueError):
        return None


def flatten_work(work, issue=None, conference_year=None, source=None):
    biblio = work.get("biblio") or {}
    refs = [
        short_openalex_id(ref)
        for ref in (work.get("referenced_works") or [])
        if short_openalex_id(ref)
    ]

    row = {
        "work_id": short_openalex_id(work.get("id")),
        "doi": normalize_doi(work.get("doi")),
        "year": work.get("publication_year"),
        "title": work.get("title") or work.get("display_name"),
        "type": work.get("type"),
        "volume": biblio.get("volume"),
        "issue": issue or biblio.get("issue"),
        "referenced_works": refs,
        "authorships": flatten_authorships(work),
        "number_of_references": len(set(refs)),
        "conference_year": conference_year,
        "source": source,
    }

    if row["conference_year"] is None:
        row["conference_year"] = volume_to_conference_year(row["volume"])

    return row


def read_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def write_jsonl(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        for row in rows:
            f.write(json.dumps(row, default=str) + "\n")


## 3 - Fetch PACMPL Works

In [ ]:
PACMPL_ISSN = "2475-1421"
pacmpl_raw_path = raw_pacmpl_dir / "pacmpl_full.jsonl"

if pacmpl_raw_path.exists() and not REFRESH_DATA:
    pacmpl_raw = read_jsonl(pacmpl_raw_path)
    print("Loaded saved PACMPL works:", len(pacmpl_raw))
else:
    query = Works().filter(primary_location={"source": {"issn": PACMPL_ISSN}})
    pacmpl_raw = []

    for page in query.paginate(per_page=200, n_max=None):
        pacmpl_raw.extend(page)

    expected = query.count()
    print("Expected:", expected)
    print("Fetched:", len(pacmpl_raw))

    write_jsonl(pacmpl_raw, pacmpl_raw_path)
    print("Saved to:", pacmpl_raw_path)


In [ ]:
pacmpl_all = pd.DataFrame([
    flatten_work(work, source="openalex_pacmpl_issn")
    for work in pacmpl_raw
])

pacmpl_master_columns = [
    "work_id",
    "doi",
    "year",
    "title",
    "type",
    "volume",
    "issue",
    "referenced_works",
    "authorships",
    "number_of_references",
]

pacmpl_filtered_columns = pacmpl_master_columns + ["conference_year"]
pldi_filtered_columns = pacmpl_filtered_columns + ["source"]
all_papers_columns = pldi_filtered_columns + ["conference"]

pacmpl_master = pacmpl_all[pacmpl_master_columns].copy()

pacmpl_master_path = raw_pacmpl_dir / "pacmpl_master.parquet"
pacmpl_master.to_parquet(pacmpl_master_path, index=False)

print(pacmpl_master.shape)
print("Saved to:", pacmpl_master_path)
display(pacmpl_master.head())


## 4 - Filter PACMPL to the Analysis Window

For PACMPL, I use `conference_year = 2016 + volume`. This is more reliable
than OpenAlex `publication_year`, because some POPL proceedings are recorded
in the previous calendar year.


In [ ]:
VALID_PACMPL_ISSUES = ["ICFP", "POPL", "OOPSLA", "OOPSLA1", "OOPSLA2"]
PAPER_TYPES = ["article", "preprint"]

EDITORIAL_RE = re.compile(
    r"\b(editorial|message from|from the editor|preface|welcome)\b",
    re.IGNORECASE,
)


def is_editorial(title):
    return isinstance(title, str) and bool(EDITORIAL_RE.search(title))


def keep_basic_paper(df):
    return (
        df["conference_year"].between(ANALYSIS_START, ANALYSIS_END)
        & df["type"].isin(PAPER_TYPES)
        & ~df["title"].apply(is_editorial)
    )


def has_reference_list(df):
    return df["number_of_references"].fillna(0).ge(1)


pacmpl_filtered = pacmpl_all[
    pacmpl_all["issue"].isin(VALID_PACMPL_ISSUES)
    & keep_basic_paper(pacmpl_all)
    & has_reference_list(pacmpl_all)
].copy()

pacmpl_filtered = (
    pacmpl_filtered
    [pacmpl_filtered_columns]
    .sort_values(["issue", "conference_year", "doi"])
    .reset_index(drop=True)
)

print(pacmpl_filtered.shape)
display(
    pacmpl_filtered
    .groupby(["issue", "conference_year"])
    .size()
    .unstack(fill_value=0)
    .reindex(VALID_PACMPL_ISSUES)
)


## 5 - Fetch PLDI 2017-2022 from DBLP and OpenAlex

In [ ]:
dblp_dir = raw_pldi_dir / "dblp"
dblp_dir.mkdir(parents=True, exist_ok=True)

openalex_doi_dir = raw_pldi_dir / "openalex_by_doi"
openalex_doi_dir.mkdir(parents=True, exist_ok=True)


def fetch_dblp_pldi_year(year):
    path = dblp_dir / f"pldi_{year}.json"

    if path.exists() and not REFRESH_DATA:
        return json.loads(path.read_text())

    url = "https://dblp.org/search/publ/api"
    params = {
        "q": f"stream:conf/pldi: year:{year}",
        "format": "json",
        "h": 500,
    }
    headers = {"User-Agent": "pc-service-citation-study/0.1"}

    for attempt in range(DBLP_MAX_RETRIES):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=30)
        except requests.exceptions.RequestException as exc:
            if attempt < DBLP_MAX_RETRIES - 1:
                wait = DBLP_RETRY_WAIT * (attempt + 1)
                print(f"DBLP connection issue for PLDI {year}: {exc}; waiting {wait}s")
                time.sleep(wait)
                continue
            raise

        if response.status_code == 429 and attempt < DBLP_MAX_RETRIES - 1:
            wait = DBLP_RETRY_WAIT * (attempt + 1)
            print(f"DBLP rate limit for PLDI {year}; waiting {wait}s")
            time.sleep(wait)
            continue

        response.raise_for_status()
        data = response.json()
        path.write_text(json.dumps(data, indent=2, default=str))
        time.sleep(DBLP_SLEEP)
        return data

    raise RuntimeError(f"Could not fetch DBLP PLDI {year}")


def pldi_dois_from_dblp(year):
    data = fetch_dblp_pldi_year(year)
    hits = data.get("result", {}).get("hits", {}).get("hit", [])

    rows = []
    for hit in hits:
        info = hit.get("info", {})

        if info.get("type") != "Conference and Workshop Papers":
            continue

        doi = normalize_doi(info.get("doi"))
        if doi is None:
            continue

        doi_suffix = doi.replace("https://doi.org/", "")
        if "." not in doi_suffix.split("/", 1)[-1]:
            continue

        rows.append({
            "conference": "PLDI",
            "conference_year": year,
            "doi": doi,
            "dblp_title": info.get("title"),
            "dblp_url": info.get("url"),
        })

    return pd.DataFrame(rows).drop_duplicates("doi")


pldi_dblp_dois = pd.concat(
    [pldi_dois_from_dblp(year) for year in range(2017, 2023)],
    ignore_index=True,
)

print(pldi_dblp_dois.shape)
display(pldi_dblp_dois.groupby("conference_year").size())
display(pldi_dblp_dois.head())

pldi_dblp_dois.to_csv(dependency_tables_dir / "pldi_dblp_dois.csv", index=False)


In [ ]:
def fetch_openalex_by_doi(doi):
    key = doi_cache_key(doi)
    path = openalex_doi_dir / f"{key}.json"

    if path.exists() and not REFRESH_DATA:
        saved = json.loads(path.read_text())
        return saved if saved else None

    try:
        work = Works()[normalize_doi(doi)]
    except Exception:
        work = None

    path.write_text(json.dumps(work, default=str))
    time.sleep(FETCH_SLEEP)
    return work


pldi_old_rows = []

for _, doi_row in pldi_dblp_dois.iterrows():
    work = fetch_openalex_by_doi(doi_row["doi"])

    if work is None:
        pldi_old_rows.append({
            "work_id": None,
            "doi": doi_row["doi"],
            "year": doi_row["conference_year"],
            "title": doi_row["dblp_title"],
            "type": None,
            "volume": None,
            "issue": "PLDI",
            "referenced_works": [],
            "authorships": [],
            "number_of_references": 0,
            "conference_year": doi_row["conference_year"],
            "source": "dblp_doi_no_oa",
        })
        continue

    row = flatten_work(
        work,
        issue="PLDI",
        conference_year=doi_row["conference_year"],
        source="dblp_doi+openalex",
    )
    row["dblp_title"] = doi_row["dblp_title"]
    row["dblp_url"] = doi_row["dblp_url"]
    pldi_old_rows.append(row)

pldi_old = pd.DataFrame(pldi_old_rows)

print(pldi_old.shape)
display(pldi_old.groupby(["source", "conference_year"]).size().unstack(fill_value=0))

missing_openalex = pldi_old[pldi_old["work_id"].isna()]
if len(missing_openalex):
    display(missing_openalex[["conference_year", "doi", "title", "source"]])
    raise ValueError("Some PLDI DOI rows did not resolve to OpenAlex works")


## 6 - Add PLDI 2023-2025 from PACMPL

In [ ]:
pldi_pacmpl = pacmpl_all[
    (pacmpl_all["issue"] == "PLDI")
    & keep_basic_paper(pacmpl_all)
    & pacmpl_all["conference_year"].between(2023, 2025)
].copy()

pldi_pacmpl["source"] = "pacmpl_master"

print(pldi_pacmpl.shape)
display(pldi_pacmpl.groupby("conference_year").size())


## 7 - Build PLDI and Combined Paper Files

In [ ]:
pldi_filtered = pd.concat([pldi_old, pldi_pacmpl], ignore_index=True, sort=False)

pldi_filtered = pldi_filtered[
    keep_basic_paper(pldi_filtered)
    & pldi_filtered["work_id"].notna()
].copy()

pldi_filtered = (
    pldi_filtered[pldi_filtered_columns]
    .sort_values(["conference_year", "doi"])
    .reset_index(drop=True)
)

pacmpl_for_all = pacmpl_filtered.copy()
pacmpl_for_all["source"] = "pacmpl_master"
pacmpl_for_all["conference"] = pacmpl_for_all["issue"]

pldi_for_all = pldi_filtered.copy()
pldi_for_all["conference"] = "PLDI"

all_papers = pd.concat([pacmpl_for_all, pldi_for_all], ignore_index=True, sort=False)
all_papers = (
    all_papers
    [all_papers_columns]
    .sort_values(["conference", "conference_year", "doi"])
    .reset_index(drop=True)
)

print("PACMPL:", pacmpl_filtered.shape)
print("PLDI:", pldi_filtered.shape)
print("All papers:", all_papers.shape)


In [ ]:
paper_counts = (
    all_papers
    .groupby(["conference", "conference_year"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["conference", "conference_year"])
)

source_counts = (
    all_papers
    .groupby(["source", "conference", "conference_year"])
    .size()
    .reset_index(name="n_papers")
    .sort_values(["source", "conference", "conference_year"])
)

display(paper_counts)
display(source_counts)


## 8 - Validate and Save

In [ ]:
if all_papers["conference_year"].min() < ANALYSIS_START:
    raise ValueError("Found papers before the analysis window")

if all_papers["conference_year"].max() > ANALYSIS_END:
    raise ValueError("Found papers after the analysis window")

if all_papers["work_id"].isna().any():
    raise ValueError("Some papers do not have OpenAlex work_id")

if list(pacmpl_master.columns) != pacmpl_master_columns:
    raise ValueError("PACMPL master columns changed")

if list(pacmpl_filtered.columns) != pacmpl_filtered_columns:
    raise ValueError("PACMPL filtered columns changed")

if list(pldi_filtered.columns) != pldi_filtered_columns:
    raise ValueError("PLDI filtered columns changed")

if list(all_papers.columns) != all_papers_columns:
    raise ValueError("Combined paper columns changed")

if ((all_papers["conference"] == "POPL") & (all_papers["conference_year"] == 2017)).any():
    raise ValueError("POPL 2017 should not appear in the PACMPL analysis sample")

duplicates = all_papers.duplicated(["work_id", "conference", "conference_year"], keep=False)
if duplicates.any():
    display(all_papers.loc[duplicates].sort_values(["work_id", "conference", "conference_year"]))
    raise ValueError("Duplicate paper rows found")

missing_refs = all_papers["referenced_works"].apply(len).eq(0).sum()
print("Papers with zero references:", missing_refs)
if missing_refs:
    display(
        all_papers.loc[
            all_papers["referenced_works"].apply(len).eq(0),
            ["conference", "conference_year", "doi", "title", "source", "number_of_references"],
        ]
    )

print("Conferences:", sorted(all_papers["conference"].unique()))
print("OOPSLA labels:", sorted(c for c in all_papers["conference"].unique() if c.startswith("OOPSLA")))


In [ ]:
pacmpl_path = prepared_pacmpl_dir / "pacmpl_filtered.parquet"
pldi_path = prepared_pldi_dir / "pldi_filtered.parquet"
all_path = prepared_all_dir / "all_papers_filtered.parquet"

pacmpl_filtered.to_parquet(pacmpl_path, index=False)
pldi_filtered.to_parquet(pldi_path, index=False)
all_papers.to_parquet(all_path, index=False)

paper_counts.to_csv(summary_tables_dir / "paper_counts_by_conference_year.csv", index=False)
source_counts.to_csv(summary_tables_dir / "paper_counts_by_source.csv", index=False)

print(pacmpl_path)
print(pldi_path)
print(all_path)
print(all_papers.shape)


## Output

| File | Description |
|---|---|
| `data/raw/pacmpl_papers/pacmpl_full.jsonl` | Raw PACMPL OpenAlex snapshot |
| `data/raw/pacmpl_papers/pacmpl_master.parquet` | Flattened PACMPL OpenAlex snapshot |
| `data/raw/pldi_papers/dblp/` | Raw DBLP responses for PLDI 2017-2022 |
| `data/raw/pldi_papers/openalex_by_doi/` | OpenAlex DOI lookups for PLDI 2017-2022 |
| `dependency_tables/pldi_dblp_dois.csv` | PLDI DOI list from DBLP |
| `data/prepared/pacmpl_papers/pacmpl_filtered.parquet` | PACMPL papers in the analysis window |
| `data/prepared/pldi_papers/pldi_filtered.parquet` | PLDI papers in the analysis window |
| `data/prepared/all_papers/all_papers_filtered.parquet` | Combined paper level dataset |
